In [1]:
!pip install -q groq sentence-transformers faiss-cpu pypdf flask pyngrok numpy pandas scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 18.8 MB/s eta 0:00:00


In [2]:
import os
import json
import pickle
import numpy as np
import pandas as pd

from flask import Flask, render_template, request, jsonify
from pyngrok import ngrok

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

from sentence_transformers import SentenceTransformer
import faiss
from pypdf import PdfReader
from groq import Groq


In [3]:
import os
from getpass import getpass

GROQ_API_KEY = os.environ.get("GROQ_API_KEY") or getpass("Enter your Groq API key: ")
client = Groq(api_key=GROQ_API_KEY)

test = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Say OK"}],
    max_tokens=10
)
print("Groq API:", test.choices[0].message.content)


Groq API: OK


In [6]:
from google.colab import files
uploaded = files.upload()

Saving who_ai.pdf to who_ai.pdf


In [7]:
def load_pdf(path: str) -> str:
    reader = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)


def split_text(text: str, chunk_size: int = 120, overlap: int = 30) -> list[str]:
    words = text.split()
    chunks, i = [], 0
    while i < len(words):
        chunk = " ".join(words[i : i + chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks


pdf_text  = load_pdf("who_ai.pdf")
documents = split_text(pdf_text)
print(f"PDF loaded → {len(documents)} overlapping chunks")

PDF loaded → 4 overlapping chunks


In [8]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

doc_embeddings = embed_model.encode(documents, convert_to_numpy=True, show_progress_bar=True)
doc_embeddings /= np.linalg.norm(doc_embeddings, axis=1, keepdims=True)

index = faiss.IndexFlatIP(doc_embeddings.shape[1])
index.add(doc_embeddings)


def retrieve(query: str, k: int = 5) -> list[str]:
    q_emb = embed_model.encode([query], convert_to_numpy=True)
    q_emb /= np.linalg.norm(q_emb, axis=1, keepdims=True)
    _, indices = index.search(q_emb, k)
    return [documents[i] for i in indices[0]]


print("FAISS index ready")
print(retrieve("heart disease risk factors")[0][:200])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS index ready
Exercise regularly * Maintain healthy weight --- [Heart Disease] What is heart disease? Heart disease refers to conditions affecting the heart and blood vessels. What are symptoms of heart disease? * 


In [9]:
from google.colab import files
uploaded = files.upload()

Saving heart.csv to heart (1).csv


In [10]:
FEATURE_NAMES = [
    "age", "sex", "cp", "trestbps", "chol", "fbs",
    "restecg", "thalach", "exang", "oldpeak", "slope", "ca", "thal"
]

df = pd.read_csv("heart.csv")
X  = df[FEATURE_NAMES]
y  = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

rf = RandomForestClassifier(
    n_estimators=200,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_s, y_train)

cv_scores = cross_val_score(rf, X_train_s, y_train, cv=5, scoring="roc_auc")
print(f"CV AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print(classification_report(y_test, rf.predict(X_test_s)))

CV AUC: 0.913 ± 0.050
              precision    recall  f1-score   support

           0       0.95      0.71      0.82        28
           1       0.80      0.97      0.88        33

    accuracy                           0.85        61
   macro avg       0.88      0.84      0.85        61
weighted avg       0.87      0.85      0.85        61



In [11]:
with open("health.pkl", "wb") as f:
    pickle.dump(rf, f)

with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Model + scaler saved")

Model + scaler saved


In [12]:
SYSTEM_PROMPT = """You are a knowledgeable, empathetic heart-health assistant built into
a cardiovascular risk tool. Your job is to help users understand their results and
general heart-health concepts.

Rules:
- Base answers on the provided medical context when relevant.
- Be concise (3-5 sentences max unless the user asks for more detail).
- Never diagnose or prescribe — always recommend seeing a doctor for personal advice.
- If a question is completely unrelated to health or the tool, politely redirect.
- Use plain English; avoid jargon unless the user clearly prefers clinical terms.
"""


def ask_llm(
    user_message: str,
    history: list,
    rag_context: str = "",
    prediction_context: str = ""
) -> str:
    enriched = user_message
    if rag_context:
        enriched = (
            f"[Relevant medical context from WHO guidelines]\n{rag_context}\n\n"
            f"[User question]\n{user_message}"
        )
    if prediction_context:
        enriched = f"[Current prediction]\n{prediction_context}\n\n" + enriched

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages += history
    messages += [{"role": "user", "content": enriched}]

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
        max_tokens=512,
        temperature=0.5
    )
    return response.choices[0].message.content.strip()


print(ask_llm("What is LDL cholesterol?", []))


LDL (Low-Density Lipoprotein) cholesterol is often called "bad" cholesterol because high levels can lead to the buildup of plaque in your arteries, increasing the risk of heart disease and stroke. It carries cholesterol from your liver to the rest of your body. Think of it like a delivery truck: if there are too many trucks on the road, they can cause traffic jams, similar to how high LDL levels can clog your arteries. To understand your specific situation, it's best to consult with a doctor who can provide personalized advice based on your overall health and test results.


In [17]:
import os

os.makedirs("templates", exist_ok=True)

html_code = """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Heart Risk Predictor</title>
<link href="https://fonts.googleapis.com/css2?family=DM+Serif+Display:ital@0;1&family=DM+Sans:wght@300;400;500;600&display=swap" rel="stylesheet">
<style>
  :root {
    --red:#E24B4A; --red-dk:#A32D2D; --teal:#1D9E75; --amber:#BA7517;
    --bg:#F4F2EC; --card:#FFFFFF; --ink:#2E2D2B;
    --muted:rgba(46,45,43,0.45); --border:rgba(46,45,43,0.1); --field:#F7F6F2;
  }
  *,*::before,*::after{box-sizing:border-box;margin:0;padding:0}
  body{font-family:'DM Sans',sans-serif;background:var(--bg);color:var(--ink);min-height:100vh}
  .page{display:flex;flex-direction:column;align-items:center;padding:3rem 1.5rem 6rem}
  .header{text-align:center;margin-bottom:2.5rem;animation:fadeDown .6s ease both}
  .eyebrow{font-size:10px;font-weight:600;letter-spacing:.22em;text-transform:uppercase;color:var(--red);margin-bottom:.9rem}
  .header h1{font-family:'DM Serif Display',serif;font-size:clamp(2.2rem,5vw,3.4rem);font-weight:400;line-height:1.1}
  .header h1 em{font-style:italic;color:var(--red)}
  .header-sub{font-size:14px;color:var(--muted);font-weight:300;max-width:360px;margin:.65rem auto 0;line-height:1.6}
  .card{width:100%;max-width:840px;background:var(--card);border:1px solid var(--border);border-radius:22px;box-shadow:0 4px 28px rgba(46,45,43,.07);overflow:hidden;animation:fadeUp .7s .1s ease both}
  .sec-label{font-size:10px;font-weight:600;letter-spacing:.17em;text-transform:uppercase;color:var(--muted);padding:1.25rem 1.75rem .7rem;border-bottom:1px solid var(--border);display:flex;align-items:center;gap:7px}
  .sec-label::before{content:'';display:block;width:6px;height:6px;border-radius:50%;background:var(--red);opacity:.7}
  .form-body{padding:1.25rem 1.75rem 1.75rem}
  .grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(200px,1fr));gap:11px;margin-bottom:1.5rem}
  .field{display:flex;flex-direction:column;gap:5px}
  .field label{font-size:10px;font-weight:600;letter-spacing:.07em;text-transform:uppercase;color:var(--muted)}
  .field label small{font-size:9px;font-weight:300;opacity:.75}
  .field input{width:100%;padding:9px 12px;border-radius:10px;border:1px solid var(--border);background:var(--field);color:var(--ink);font-family:'DM Sans',sans-serif;font-size:13px;outline:none;transition:border-color .2s,background .2s}
  .field input:focus{border-color:rgba(226,75,74,.5);background:#fff}
  .btn{width:100%;padding:13px;border-radius:11px;border:none;background:var(--red);color:#fff;font-family:'DM Sans',sans-serif;font-size:13px;font-weight:600;letter-spacing:.06em;text-transform:uppercase;cursor:pointer;transition:opacity .2s,transform .15s}
  .btn:hover{opacity:.87;transform:translateY(-1px)}
  .divider{height:1px;background:var(--border);margin:0 1.75rem}
  .result{padding:1.5rem 1.75rem 1.75rem}
  .result-title{font-family:'DM Serif Display',serif;font-size:1.5rem;font-weight:400;margin-bottom:1.1rem}
  .risk-meta{display:flex;justify-content:space-between;align-items:baseline;margin-bottom:9px}
  .risk-label{font-size:10px;text-transform:uppercase;letter-spacing:.1em;color:var(--muted);font-weight:500}
  .risk-pct{font-family:'DM Serif Display',serif;font-size:1.9rem;line-height:1}
  .risk-pct small{font-family:'DM Sans',sans-serif;font-size:12px;color:var(--muted);font-weight:300;margin-left:2px}
  .track{height:7px;background:rgba(46,45,43,.08);border-radius:100px;overflow:hidden;margin-bottom:5px}
  .fill{height:100%;width:0%;border-radius:100px;background:linear-gradient(90deg,var(--teal) 0%,var(--amber) 55%,var(--red) 100%);transition:width 1.1s cubic-bezier(.4,0,.2,1)}
  .scale{display:flex;justify-content:space-between;margin-bottom:1rem}
  .scale span{font-size:9px;color:var(--muted);letter-spacing:.04em}
  .chips{display:flex;gap:7px;flex-wrap:wrap;margin-bottom:1rem}
  .chip{padding:4px 13px;border-radius:100px;font-size:11px;font-weight:500}
  .chip-low{background:rgba(29,158,117,.1);color:#0F6E56;border:1px solid rgba(29,158,117,.25)}
  .chip-mid{background:rgba(186,117,23,.1);color:#7A4E0C;border:1px solid rgba(186,117,23,.25)}
  .chip-high{background:rgba(226,75,74,.1);color:var(--red-dk);border:1px solid rgba(226,75,74,.25)}
  .insight{font-size:13px;line-height:1.65;color:var(--ink);background:var(--field);border-radius:10px;padding:12px 14px;border:1px solid var(--border)}
  .disclaimer{margin-top:2.5rem;text-align:center;font-size:11px;color:var(--muted);line-height:1.7;max-width:480px}
  #chat-btn{position:fixed;bottom:26px;right:26px;z-index:100;width:54px;height:54px;border-radius:50%;border:none;background:var(--red);color:#fff;cursor:pointer;box-shadow:0 4px 18px rgba(226,75,74,.35);display:flex;align-items:center;justify-content:center;transition:transform .2s}
  #chat-btn:hover{transform:scale(1.07)}
  #chat-win{position:fixed;bottom:92px;right:26px;z-index:99;width:340px;max-height:520px;background:#fff;border:1px solid var(--border);border-radius:20px;box-shadow:0 8px 40px rgba(46,45,43,.13);display:none;flex-direction:column;overflow:hidden}
  #chat-win.open{display:flex;animation:fadeUp .22s ease}
  .chat-hdr{background:var(--red);padding:12px 16px;display:flex;align-items:center;gap:9px}
  .chat-hdr-ico{width:28px;height:28px;background:rgba(255,255,255,.2);border-radius:50%;display:flex;align-items:center;justify-content:center;flex-shrink:0}
  .chat-hdr h3{font-size:13px;font-weight:600;color:#fff;line-height:1.2}
  .chat-hdr p{font-size:10px;color:rgba(255,255,255,.7)}
  #chat-msgs{flex:1;overflow-y:auto;padding:12px 12px 6px;display:flex;flex-direction:column;gap:8px;background:#F7F6F2;scroll-behavior:smooth}
  .msg{max-width:86%;padding:8px 12px;border-radius:13px;font-size:13px;line-height:1.55}
  .msg.bot{background:#fff;color:var(--ink);border:1px solid var(--border);border-bottom-left-radius:3px;align-self:flex-start}
  .msg.user{background:var(--red);color:#fff;border-bottom-right-radius:3px;align-self:flex-end}
  .msg.typing{background:#fff;border:1px solid var(--border);color:var(--muted);align-self:flex-start;font-style:italic;font-size:12px}
  .chat-row{display:flex;gap:7px;padding:9px 11px;background:#fff;border-top:1px solid var(--border)}
  #chat-in{flex:1;border:1px solid var(--border);border-radius:9px;padding:8px 11px;font-size:13px;font-family:'DM Sans',sans-serif;color:var(--ink);background:var(--field);outline:none;resize:none}
  #chat-in:focus{border-color:rgba(226,75,74,.45);background:#fff}
  #chat-send{width:34px;height:34px;border-radius:9px;border:none;background:var(--red);color:#fff;cursor:pointer;display:flex;align-items:center;justify-content:center;flex-shrink:0;transition:opacity .2s;align-self:flex-end}
  #chat-send:hover{opacity:.85}
  @keyframes fadeDown{from{opacity:0;transform:translateY(-16px)}to{opacity:1;transform:translateY(0)}}
  @keyframes fadeUp{from{opacity:0;transform:translateY(18px)}to{opacity:1;transform:translateY(0)}}
  @media(max-width:560px){.form-body,.result{padding:1.1rem 1.2rem 1.4rem}.sec-label{padding:1.1rem 1.2rem .65rem}.divider{margin:0 1.2rem}.grid{grid-template-columns:1fr 1fr}#chat-win{width:calc(100vw - 32px);right:16px}}
  @media(max-width:380px){.grid{grid-template-columns:1fr}}
</style>
</head>
<body>
<div class="page">
  <header class="header">
    <div class="eyebrow">AI Clinical Assessment</div>
    <h1>Heart Risk<br><em>Predictor</em></h1>
    <p class="header-sub">Enter patient clinical parameters for an AI-powered cardiovascular risk assessment.</p>
  </header>
  <div class="card">
    <div class="sec-label">Patient Clinical Parameters</div>
    <form action="/predict" method="POST">
      <div class="form-body">
        <div class="grid">
          {% set fields = [
            ('age','Age','years','54',1,120,1),
            ('sex','Sex','0=F · 1=M','1',0,1,1),
            ('cp','Chest Pain Type','0–3','2',0,3,1),
            ('trestbps','Resting BP','mm Hg','130',80,200,1),
            ('chol','Cholesterol','mg/dl','220',100,600,1),
            ('fbs','Fasting Blood Sugar','>120 mg','0',0,1,1),
            ('restecg','Resting ECG','0–2','1',0,2,1),
            ('thalach','Max Heart Rate','bpm','150',60,220,1),
            ('exang','Exercise Angina','0=No 1=Yes','0',0,1,1),
            ('oldpeak','ST Depression','Oldpeak','1.0',0,7,0.1),
            ('slope','ST Slope','0–2','1',0,2,1),
            ('ca','Major Vessels','0–4','0',0,4,1),
            ('thal','Thalassemia','1–3','2',1,3,1)
          ] %}
          {% for name,label,hint,ph,mn,mx,stp in fields %}
          <div class="field">
            <label for="{{ name }}">{{ label }} <small>{{ hint }}</small></label>
            <input type="number" id="{{ name }}" name="{{ name }}"
                   placeholder="{{ ph }}" min="{{ mn }}" max="{{ mx }}" step="{{ stp }}" required>
          </div>
          {% endfor %}
        </div>
        <button type="submit" class="btn">Run Prediction &rarr;</button>
      </div>
    </form>
    {% if prediction_text %}
    <div class="divider"></div>
    <div class="sec-label">Assessment Result</div>
    <div class="result">
      <p class="result-title">{{ prediction_text }}</p>
      <div class="risk-meta">
        <span class="risk-label">Cardiovascular Risk Score</span>
        <div class="risk-pct">{{ risk }}<small>%</small></div>
      </div>
      <div class="track"><div class="fill" id="fill"></div></div>
      <div class="scale"><span>Low</span><span>Moderate</span><span>High</span></div>
      <div class="chips">
        {% if risk|int < 30 %}
          <span class="chip chip-low">Low Risk</span>
          <span class="chip chip-low">Routine Monitoring</span>
        {% elif risk|int < 65 %}
          <span class="chip chip-mid">Moderate Risk</span>
          <span class="chip chip-mid">Follow-Up Recommended</span>
        {% else %}
          <span class="chip chip-high">High Risk</span>
          <span class="chip chip-high">Urgent Review Advised</span>
        {% endif %}
      </div>
      {% if insight %}<div class="insight">{{ insight }}</div>{% endif %}
    </div>
    {% endif %}
  </div>
  <p class="disclaimer"><strong>Clinical disclaimer.</strong> This tool provides a statistical estimate and is not a substitute for professional medical diagnosis. Always consult a qualified cardiologist.</p>
</div>

<button id="chat-btn" title="Ask a health question">
  <svg width="22" height="22" viewBox="0 0 24 24" fill="none">
    <path d="M21 15c0 .53-.21 1.04-.59 1.41-.37.38-.88.59-1.41.59H7l-4 4V5c0-.53.21-1.04.59-1.41C3.96 3.21 4.47 3 5 3h14c.53 0 1.04.21 1.41.59.38.37.59.88.59 1.41v10z" stroke="white" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"/>
  </svg>
</button>

<div id="chat-win">
  <div class="chat-hdr">
    <div class="chat-hdr-ico">
      <svg width="15" height="15" viewBox="0 0 32 32" fill="none">
        <path d="M16 28S3 20.5 3 11.5C3 7.36 6.13 4 10 4c2.35 0 4.43 1.21 6 3 1.57-1.79 3.65-3 6-3C25.87 4 29 7.36 29 11.5 29 20.5 16 28 16 28z" fill="white"/>
      </svg>
    </div>
    <div>
      <h3>Heart Health Assistant</h3>
    </div>
  </div>
  <div id="chat-msgs">
    <div class="msg bot">Hi! I can answer questions about heart health, risk factors, medications, and your results. What would you like to know?</div>
  </div>
  <div class="chat-row">
    <textarea id="chat-in" rows="1" placeholder="Ask about cholesterol, BP, your results…"></textarea>
    <button id="chat-send">
      <svg width="15" height="15" viewBox="0 0 24 24" fill="none">
        <path d="M22 2L11 13M22 2l-7 20-4-9-9-4 20-7z" stroke="white" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"/>
      </svg>
    </button>
  </div>
</div>

<script>
  var risk = {{ risk if risk else 0 }};
  var fill = document.getElementById('fill');
  if (fill) setTimeout(() => fill.style.width = risk + '%', 300);

  document.getElementById('chat-btn').onclick = () =>
    document.getElementById('chat-win').classList.toggle('open');

  var chatHistory = [];
  var inp = document.getElementById('chat-in');
  inp.addEventListener('input', () => {
    inp.style.height = 'auto';
    inp.style.height = Math.min(inp.scrollHeight, 90) + 'px';
  });

  function sendMsg() {
    var text = inp.value.trim();
    if (!text) return;
    var msgs = document.getElementById('chat-msgs');
    var u = document.createElement('div');
    u.className = 'msg user'; u.textContent = text;
    msgs.appendChild(u);
    var t = document.createElement('div');
    t.className = 'msg typing'; t.id = 'typing'; t.textContent = 'Thinking…';
    msgs.appendChild(t);
    msgs.scrollTop = msgs.scrollHeight;
    inp.value = ''; inp.style.height = 'auto';
    fetch('/chat', {
      method: 'POST',
      headers: {'Content-Type': 'application/json'},
      body: JSON.stringify({ message: text, history: chatHistory })
    })
    .then(r => r.json())
    .then(data => {
      document.getElementById('typing')?.remove();
      var b = document.createElement('div');
      b.className = 'msg bot';
      b.textContent = data.reply || 'Sorry, I could not find an answer.';
      msgs.appendChild(b);
      msgs.scrollTop = msgs.scrollHeight;
      chatHistory.push({role:'user', content: text});
      chatHistory.push({role:'assistant', content: data.reply});
      if (chatHistory.length > 20) chatHistory = chatHistory.slice(-20);
    })
    .catch(() => {
      document.getElementById('typing')?.remove();
      var e = document.createElement('div');
      e.className = 'msg bot'; e.textContent = 'Connection error — please try again.';
      msgs.appendChild(e);
      msgs.scrollTop = msgs.scrollHeight;
    });
  }

  document.getElementById('chat-send').onclick = sendMsg;
  inp.addEventListener('keydown', e => {
    if (e.key === 'Enter' && !e.shiftKey) { e.preventDefault(); sendMsg(); }
  });
</script>
</body>
</html>
"""


with open('templates/index.html', 'w') as f:
    f.write(html_code)

print('Template written')

Template written


In [18]:
app = Flask(__name__)

FIELD_RANGES = {
    "age":      (1,   120),
    "sex":      (0,   1),
    "cp":       (0,   3),
    "trestbps": (80,  200),
    "chol":     (100, 600),
    "fbs":      (0,   1),
    "restecg":  (0,   2),
    "thalach":  (60,  220),
    "exang":    (0,   1),
    "oldpeak":  (0,   7),
    "slope":    (0,   2),
    "ca":       (0,   4),
    "thal":     (1,   3),
}


def parse_and_validate(form_data):
    values, errors = [], []
    for col in FEATURE_NAMES:
        raw = form_data.get(col, "").strip()
        try:
            val = float(raw)
        except ValueError:
            errors.append(f"{col}: must be a number")
            values.append(0.0)
            continue
        lo, hi = FIELD_RANGES.get(col, (-1e9, 1e9))
        if not (lo <= val <= hi):
            errors.append(f"{col}: {val} out of range [{lo}, {hi}]")
        values.append(val)
    return values, errors


def feature_summary(values):
    labels = {
        "age":"Age","sex":"Sex","cp":"Chest pain type",
        "trestbps":"Resting BP","chol":"Cholesterol","fbs":"Fasting blood sugar>120",
        "restecg":"Rest ECG","thalach":"Max heart rate","exang":"Exercise angina",
        "oldpeak":"ST depression","slope":"ST slope","ca":"Major vessels","thal":"Thalassemia"
    }
    return ", ".join(f"{labels[k]}={v}" for k, v in zip(FEATURE_NAMES, values))


@app.route("/")
def home():
    return render_template("index.html", risk=0)


@app.route("/predict", methods=["POST"])
def predict():
    form = request.form.to_dict()
    user_input, errors = parse_and_validate(form)

    if errors:
        return render_template(
            "index.html", risk=0,
            prediction_text="Input error: " + " | ".join(errors)
        )

    scaled = scaler.transform([user_input])
    pred   = rf.predict(scaled)[0]
    prob   = round(rf.predict_proba(scaled)[0][1] * 100, 1)
    result = "Heart Disease Detected" if pred == 1 else "No Heart Disease Detected"

    feat_str = feature_summary(user_input)
    pred_ctx = f"Prediction: {result} | Risk score: {prob}% | Patient data: {feat_str}"
    rag_ctx  = "\n".join(retrieve("heart disease risk factors causes", k=4))

    try:
        insight = ask_llm(
            user_message=(
                "In 2-3 sentences, explain what the prediction result means for this "
                "patient and the key factors that likely drove it. Be direct and clear."
            ),
            history=[],
            rag_context=rag_ctx,
            prediction_context=pred_ctx
        )
    except Exception as e:
        insight = f"(AI insight unavailable: {e})"

    return render_template(
        "index.html",
        prediction_text=result,
        risk=prob,
        insight=insight
    )


@app.route("/chat", methods=["POST"])
def chat():
    data     = request.get_json(force=True)
    user_msg = data.get("message", "").strip()
    history  = data.get("history", [])

    if not user_msg:
        return jsonify(reply="Please ask a question.")

    rag_ctx = "\n".join(retrieve(user_msg, k=5))

    try:
        reply = ask_llm(
            user_message=user_msg,
            history=history,
            rag_context=rag_ctx
        )
    except Exception as e:
        reply = f"Something went wrong: {e}"

    return jsonify(reply=reply)


print("Flask app defined")

Flask app defined


In [19]:
import os
from getpass import getpass

NGROK_AUTH_TOKEN = os.environ.get("NGROK_AUTH_TOKEN") or getpass("Enter your ngrok auth token: ")
!ngrok config add-authtoken "$NGROK_AUTH_TOKEN"

public_url = ngrok.connect(5000)
print("App live at:", public_url)

app.run(port=5000)


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
App live at: NgrokTunnel: "https://1277-35-252-69-213.ngrok-free.app" -> "http://localhost:5000"
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [25/Jun/2026 09:18:47] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [25/Jun/2026 09:18:49] "GET /favicon.ico HTTP/1.1" 404 -
